# Notebook 03b — Investigación de los códigos de provincia internacionales

## Objetivo

Identificar a qué país corresponde cada código de provincia distinto presente en la cartera internacional de Selmark, a partir de la información complementaria disponible en la tabla `silver.dim_cliente` (localidad, dirección, nombre del cliente, patrón del código postal).

Esta investigación alimenta la decisión metodológica 2.1 del informe de decisiones y prepara el mapeo de países que se aplicará en la sección de análisis comercial de la cartera internacional.

In [1]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb


## 1. Listado completo de códigos de provincia en cartera internacional

Se obtienen todos los códigos de provincia que aparecen en clientes clasificados como INTERNACIONAL, ordenados por número de clientes para priorizar los más relevantes.

In [2]:
codigos_internacionales = con.execute("""
    SELECT 
        codigo_provincia_cliente,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
    GROUP BY codigo_provincia_cliente
    ORDER BY num_clientes DESC
""").fetchdf()

print(f"Códigos distintos en cartera internacional: {len(codigos_internacionales)}")
print(f"Total clientes internacionales: {codigos_internacionales['num_clientes'].sum()}\n")
print(codigos_internacionales.to_string(index=False))

Códigos distintos en cartera internacional: 159
Total clientes internacionales: 1399

codigo_provincia_cliente  num_clientes
                     410           442
                     231           256
                     000            84
                     471            59
                      NO            56
                     CAN            55
                      NL            32
                     NaN            27
                     299            26
                     385            14
                     360            12
                     213            12
                     216            12
                     357             9
                     609             8
                     398             7
                     316             7
                      RU             7
                     367             6
                     382             6
                     041             6
                     305             5
                 

## 2. Investigación por código de provincia

Para cada código, se muestran las localidades únicas de los clientes que lo tienen asignado. Los nombres de las localidades son el indicador más fiable para identificar el país, ya que la mayoría de las ciudades son reconocibles.

In [3]:
investigacion = con.execute("""
    SELECT 
        codigo_provincia_cliente,
        COUNT(*) AS num_clientes,
        STRING_AGG(DISTINCT localidad_cliente, ' | ') AS localidades_distintas
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
    GROUP BY codigo_provincia_cliente
    ORDER BY num_clientes DESC
""").fetchdf()

# Mostrar bonito, código a código
for _, fila in investigacion.iterrows():
    print(f"\n{'='*70}")
    print(f"CÓDIGO: {fila['codigo_provincia_cliente']}  ·  {fila['num_clientes']} clientes")
    print(f"{'='*70}")
    localidades = fila['localidades_distintas'].split(' | ') if fila['localidades_distintas'] else []
    # Mostrar las primeras 20 localidades para no saturar
    for loc in localidades[:20]:
        print(f"  · {loc}")
    if len(localidades) > 20:
        print(f"  ... y {len(localidades) - 20} localidades más")


CÓDIGO: 410  ·  442 clientes
  · CANALICCHIO TREMESTIERI ETNEO
  · MILANO
  · ALASSIO
  · SAN SALVO CH
  · SASSARI
  · SCIACCA
  · LOC. SALVATERRA
  · FLERO
  · BARLETTA
  · CELLE LIGURE
  · CONVERSANO
  · LECCE
  · VESTONE
  · TORTONA
  · SAVONA
  · GUASTALLA
  · BORGONOVO V.T.
  · CALTANISETTA
  · MUGNANO DI NAPOLI
  · ALCAMO
  ... y 322 localidades más

CÓDIGO: 231  ·  256 clientes
  · FELGUEIRAS
  · COSTA DA CAPARICA
  · AMARANTE - LIXA
  · PAÇOS DE FERREIRA
  · VILA NOVA DE FAMALICAO-VILAR.C
  · LAGOS
  · BATALHA
  · QUARTEIRA LOULÉ
  · LAMEGO
  · MIRANDELA
  · COIMBRA
  · SEIXAL - CORROIOS
  · MONTE GORDO
  · FIGUEIRA DA FOZ
  · SANTA MARIA DA FEIRA - FIAES
  · LOULE
  · PERAFITA - MATOSINHOS
  · CALDAS DA RAINHA
  · PESSEGUEIRO DO VOUGA
  · ALMADA
  ... y 170 localidades más

CÓDIGO: 000  ·  84 clientes
  · CHOTOMOW
  · Zielona Gora
  · PLEWISKA
  · SRODA WIELKOPOLSKA
  · BIALYSTOK
  · SWIERKLANIEC
  · WROCLAW
  · ZAKOPANE
  · OLKUSZ
  · ZIELONA GORA
  · Plonsk
  · WIERZCHOSLAW

## 3. Pistas adicionales: patrones del código postal

El formato del código postal aporta otra pista valiosa, ya que cada país tiene su propio sistema (5 dígitos en Italia, formato XX-XXX en Polonia, formato LDL DLD en Canadá, etc.).

In [4]:
patrones_cp = con.execute("""
    SELECT 
        codigo_provincia_cliente,
        codigo_postal_original,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
    GROUP BY codigo_provincia_cliente, codigo_postal_original
    ORDER BY codigo_provincia_cliente, num_clientes DESC
""").fetchdf()

# Mostrar muestra por código
codigos_unicos = patrones_cp['codigo_provincia_cliente'].unique()
for codigo in codigos_unicos:
    subset = patrones_cp[patrones_cp['codigo_provincia_cliente'] == codigo].head(5)
    print(f"\nCódigo {codigo} — ejemplos de CP:")
    for _, fila in subset.iterrows():
        print(f"  · CP: {fila['codigo_postal_original']:<15}  ({fila['num_clientes']} cliente/s)")


Código 000 — ejemplos de CP:
  · CP: 84-141           (3 cliente/s)
  · CP: 05-110           (2 cliente/s)
  · CP: 63-000           (2 cliente/s)
  · CP: 05-200           (2 cliente/s)
  · CP: 87-100           (2 cliente/s)

Código 001 — ejemplos de CP:
  · CP: HA7B1G           (1 cliente/s)
  · CP: WN1 2PU          (1 cliente/s)

Código 003 — ejemplos de CP:
  · CP: 099              (1 cliente/s)

Código 036 — ejemplos de CP:
  · CP: null             (1 cliente/s)
  · CP: 28045            (1 cliente/s)

Código 040 — ejemplos de CP:
  · CP: 4063             (2 cliente/s)

Código 041 — ejemplos de CP:
  · CP: 24400            (3 cliente/s)
  · CP: 24402            (1 cliente/s)
  · CP: 24540            (1 cliente/s)
  · CP: 24300            (1 cliente/s)

Código 08 — ejemplos de CP:
  · CP: 06800            (1 cliente/s)

Código 100 — ejemplos de CP:
  · CP: 05348            (1 cliente/s)

Código 197 — ejemplos de CP:
  · CP: NJ 07724         (1 cliente/s)
  · CP: 91502            (1 c

## 4. Inspección detallada por código (caso por caso)

Para los códigos que no se identifiquen claramente con las pistas anteriores, se inspeccionan algunos clientes concretos con su nombre, dirección y CP, lo cual suele ser definitivo.

In [5]:
def investigar_codigo(codigo):
    """Muestra clientes representativos de un código de provincia."""
    print(f"\n{'='*70}")
    print(f"INSPECCIÓN DETALLADA DEL CÓDIGO: {codigo}")
    print(f"{'='*70}")
    
    detalle = con.execute("""
        SELECT 
            nombre_cliente,
            direccion_cliente,
            localidad_cliente,
            codigo_postal_original
        FROM silver.dim_cliente
        WHERE codigo_provincia_cliente = ?
        ORDER BY nombre_cliente
        LIMIT 10
    """, [codigo]).fetchdf()
    
    for _, fila in detalle.iterrows():
        print(f"\n  Cliente:   {fila['nombre_cliente']}")
        print(f"  Dirección: {fila['direccion_cliente']}")
        print(f"  Localidad: {fila['localidad_cliente']}")
        print(f"  CP:        {fila['codigo_postal_original']}")

# Investigar los códigos sin identificar
investigar_codigo('471')
investigar_codigo('231')


INSPECCIÓN DETALLADA DEL CÓDIGO: 471

  Cliente:   AN BVBA
  Dirección: BREDABAAN 180
  Localidad: BRASSCHAAT
  CP:        2930

  Cliente:   ANNELIES DUIJVESTEIN
  Dirección: Geestbrugweg 19C
  Localidad: Rijswijk
  CP:        2281 CC

  Cliente:   BUSTELLI LINGERIE
  Dirección: STAD 14/2
  Localidad: HAMONT
  CP:        3930

  Cliente:   CHALINVEST BVBA
  Dirección: WARMOEZENIERSSTRAAT-32
  Localidad: BRUGGE
  CP:        8000

  Cliente:   CHRISTEL UYDENS
  Dirección: DORP, 4
  Localidad: RIJKEVORSEL
  CP:        2310

  Cliente:   COUTURE RACHEL BVBA
  Dirección: AUG. WAUTERSTRAAT 11 11
  Localidad: TEMSE
  CP:        9140

  Cliente:   D.E.W. bvba
  Dirección: MOLENSTRAAT 2
  Localidad: NAZARETH
  CP:        9810

  Cliente:   DE LODDERE ILSE
  Dirección: POLENPLEIN 29
  Localidad: ARDOOIE
  CP:        8850

  Cliente:   DESSOUS D. BVBA
  Dirección: MEIRHOEVEDREEF, 11
  Localidad: GROBBENDONK
  CP:        2280

  Cliente:   DOL & FIJN - UYTTERSPROT BRENDA
  Dirección: NIEUWBAAN 7

## 5. Construcción de la tabla de mapeo de países

A partir de la investigación anterior, se construye una tabla auxiliar `silver.mapeo_paises` que asocia cada código de provincia internacional con su país correspondiente. Esta tabla se utilizará en la sección de análisis de la cartera internacional del proyecto.

La asignación se realiza por evidencia (localidades reconocibles y formatos de código postal). Los códigos sin clasificar (un caso aislado: 558) y los códigos con clientes españoles que no se filtraron correctamente se marcan explícitamente para trazabilidad.

In [6]:
print("Creando silver.mapeo_paises...\n")

con.execute("""
    CREATE OR REPLACE TABLE silver.mapeo_paises AS
    SELECT codigo, pais
    FROM (VALUES
        -- ITALIA (códigos 3XX completos + 4XX bajo)
        ('301','Italia'),('302','Italia'),('305','Italia'),('307','Italia'),('308','Italia'),
        ('309','Italia'),('310','Italia'),('311','Italia'),('312','Italia'),('314','Italia'),
        ('315','Italia'),('316','Italia'),('322','Italia'),('324','Italia'),('325','Italia'),
        ('327','Italia'),('332','Italia'),('333','Italia'),('334','Italia'),('335','Italia'),
        ('338','Italia'),('339','Italia'),('341','Italia'),('343','Italia'),('345','Italia'),
        ('346','Italia'),('347','Italia'),('348','Italia'),('349','Italia'),('350','Italia'),
        ('351','Italia'),('352','Italia'),('357','Italia'),('358','Italia'),('359','Italia'),
        ('360','Italia'),('361','Italia'),('362','Italia'),('366','Italia'),('367','Italia'),
        ('368','Italia'),('369','Italia'),('370','Italia'),('371','Italia'),('373','Italia'),
        ('374','Italia'),('376','Italia'),('380','Italia'),('382','Italia'),('385','Italia'),
        ('386','Italia'),('387','Italia'),('388','Italia'),('389','Italia'),('390','Italia'),
        ('392','Italia'),('393','Italia'),('395','Italia'),('396','Italia'),('397','Italia'),
        ('398','Italia'),('399','Italia'),('400','Italia'),('401','Italia'),('402','Italia'),
        ('403','Italia'),('406','Italia'),('408','Italia'),('410','Italia'),
        -- PORTUGAL
        ('201','Portugal'),('203','Portugal'),('205','Portugal'),('206','Portugal'),('207','Portugal'),
        ('208','Portugal'),('209','Portugal'),('210','Portugal'),('211','Portugal'),('212','Portugal'),
        ('213','Portugal'),('214','Portugal'),('215','Portugal'),('216','Portugal'),('220','Portugal'),
        ('231','Portugal'),('299','Portugal'),
        -- BÉLGICA
        ('465','Bélgica'),('469','Bélgica'),('471','Bélgica'),
        -- PAÍSES BAJOS
        ('460','Países Bajos'),('466','Países Bajos'),('609','Países Bajos'),
        ('NL','Países Bajos'),('NH','Países Bajos'),('ZH','Países Bajos'),('GR','Países Bajos'),
        -- POLONIA / RESTO EUROPA
        ('000','Polonia'),
        ('001','Reino Unido'),('605','Reino Unido'),('611','Reino Unido'),
        ('040','Austria'),
        ('440','Alemania'),('NRW','Alemania'),('DE','Alemania'),
        ('257','Andorra'),('258','Andorra'),
        ('674','San Marino'),
        ('NO','Noruega'),('FR','Francia'),('FI','Finlandia'),('DK','Dinamarca'),
        ('LT','Lituania'),('EE','Estonia'),('CZ','República Checa'),('RO','Rumanía'),
        ('SI','Eslovenia'),('LU','Luxemburgo'),('CY','Chipre'),('MT','Malta'),
        ('MD','Moldavia'),('RU','Rusia'),('HE','Grecia'),
        ('LI','Bélgica/Países Bajos'),
        -- AMÉRICA
        ('CAN','Canadá'),('MX','México'),('100','México'),
        ('CL','Chile'),('EC','Ecuador'),('DO','República Dominicana'),
        ('197','Estados Unidos'),
        -- ASIA / OCEANÍA
        ('CN','China'),('HK','Hong Kong'),('TW','Taiwán'),('KR','Corea del Sur'),
        ('IN','India'),('PK','Pakistán'),('KG','Kirguistán'),
        ('IL','Israel'),('AE','Emiratos Árabes Unidos'),('QA','Qatar'),
        ('KW','Kuwait'),('LB','Líbano'),('53','Líbano'),
        ('AU','Australia'),
        -- ÁFRICA
        ('MA','Marruecos'),('TN','Túnez'),('GH','Ghana'),('003','Senegal'),
        ('ZA','Sudáfrica'),('SUD','Sudáfrica'),
        -- ESPAÑOLES MAL CLASIFICADOS (errores de carga en ERP)
        ('33','España (error carga)'),('46','España (error carga)'),
        ('08','España (error carga)'),('036','España (error carga)'),('36','España (error carga)'),
        -- SIN IDENTIFICAR (1 cliente, sin pistas claras)
        ('558','Por identificar')
    ) AS t(codigo, pais)
""")

n = con.execute("SELECT COUNT(*) FROM silver.mapeo_paises").fetchone()[0]
print(f"✅ Tabla silver.mapeo_paises creada con {n} entradas")

Creando silver.mapeo_paises...

✅ Tabla silver.mapeo_paises creada con 156 entradas


In [7]:
print("=" * 70)
print("COBERTURA DEL MAPEO SOBRE LA CARTERA INTERNACIONAL")
print("=" * 70)

resultado = con.execute("""
    SELECT 
        COALESCE(m.pais, 'SIN MAPEAR') AS pais,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente d
    LEFT JOIN silver.mapeo_paises m ON d.codigo_provincia_cliente = m.codigo
    WHERE d.tipo_mercado = 'INTERNACIONAL'
    GROUP BY pais
    ORDER BY num_clientes DESC
""").fetchdf()

print(resultado.to_string(index=False))
print(f"\nTotal clientes internacionales: {resultado['num_clientes'].sum()}")

COBERTURA DEL MAPEO SOBRE LA CARTERA INTERNACIONAL
                  pais  num_clientes
                Italia           632
              Portugal           337
               Polonia            84
               Bélgica            61
               Noruega            56
                Canadá            55
          Países Bajos            46
            SIN MAPEAR            37
  España (error carga)            10
                 Rusia             7
           Reino Unido             6
              Alemania             5
                México             4
               Francia             3
                 China             2
                Chipre             2
            Luxemburgo             2
               Austria             2
             Sudáfrica             2
                 Chile             2
             Hong Kong             2
                 Malta             2
              Moldavia             2
                 India             2
            San Marino  

## 6. Investigación de los clientes SIN MAPEAR (códigos null o no identificados)

Los clientes que no han podido asignarse a un país a través del código de provincia se investigan mediante el campo `localidad_cliente`. Se utiliza un mapeo de ciudades reconocibles a países para resolver la mayoría de los casos.

In [8]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"
con = duckdb.connect(str(RUTA_DUCKDB))

print("=" * 70)
print("LOCALIDADES DE LOS CLIENTES SIN MAPEAR")
print("=" * 70)

sin_mapear = con.execute("""
    SELECT 
        UPPER(TRIM(d.localidad_cliente)) AS localidad_upper,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente d
    LEFT JOIN silver.mapeo_paises m ON d.codigo_provincia_cliente = m.codigo
    WHERE d.tipo_mercado = 'INTERNACIONAL' AND m.pais IS NULL
    GROUP BY localidad_upper
    ORDER BY num_clientes DESC
""").fetchdf()

print(f"Localidades distintas sin mapear: {len(sin_mapear)}")
print(f"Total clientes sin mapear: {sin_mapear['num_clientes'].sum()}\n")
print(sin_mapear.to_string(index=False))

LOCALIDADES DE LOS CLIENTES SIN MAPEAR
Localidades distintas sin mapear: 33
Total clientes sin mapear: 37

       localidad_upper  num_clientes
            PONFERRADA             4
                LISBOA             2
             BOOISCHOT             1
                HULSTE             1
              WINKSELE             1
VILA MOURA - QUARTEIRA             1
             PURMEREND             1
                BILZEN             1
              BEMBIBRE             1
                BINCHE             1
           IJSSELSTEIN             1
           DENDERLEEUW             1
                TIENEN             1
              MARBELLA             1
         PUURS-KALFORT             1
              SABADELL             1
       QUINTA DO CONDE             1
       RIBES DE FRESER             1
                  ROMA             1
                MONÇAO             1
                 TEMSE             1
                  NULL             1
                 FÜRTH             1
     

## 7. Clasificación automática de localidades sin mapear

Se construye una clasificación de las localidades únicas detectadas en clientes SIN MAPEAR, agrupándolas por país según el reconocimiento de cada ciudad. Esta clasificación se utilizará para refinar la lógica de `tipo_mercado` en `silver.dim_cliente` con una regla adicional basada en localidad.

Los casos ambiguos (localidades que pueden existir en varios países, como MONÇÃO) se marcan para revisión manual.

In [9]:
# ============================================================
# DICCIONARIO DE LOCALIDADES → PAÍS (basado en el listado de 178)
# ============================================================

# ESPAÑA — ciudades y municipios reconocibles
localidades_es = {
    # Galicia
    'VIGO', 'A CORUÑA', 'OURENSE', 'PONTEAREAS', 'CARBALLO', 'BURELA',
    'ORDENES', 'PONTECESO', 'MOAÑA', 'REDONDELA', 'SANTIAGO DE COMPOSTELA',
    'NAVIA', 'VIMIANZO', 'VILLABONA', 'VILLAGARCIA', 'JUNQUERA CEDEIRA',
    'SANXENXO', 'PORTO DO SON', 'O PORRIÑO', 'XINZO DE LIMIA', 'PORRIÑO',
    'MONFORTE DE LEMOS', 'NARON', 'CHAPELA(SAN FAUSFO)', 'VILAVERDE(ZAMANES)',
    'ARTEIXO', 'VILAGARCIA', 'TORROSO(SAN MAMENDE)', 'RIBADEO',
    # Asturias
    'GIJON', 'POLA DE LAVIANA', 'POLA DE SIERO', 'MIERES', 'COLUNGA',
    'ARRIONDAS', 'EL ENTREGO', 'VILLAVICIOSA',
    # Castilla y León
    'PONFERRADA', 'ZAMORA', 'TORO', 'MEDINA DEL CAMPO', 'MIRANDA DE EBRO',
    'BURGOS', 'BRIVIESCA', 'PALENCIA', 'GUARDO', 'CALANDA', 'GUIJUELO',
    'BEMBIBRE', 'FUENTES DE ROPEL - ZAMORA', 'ARANDA DE DUERO',
    'VILLALON DE CAMPOS', 'CACABELOS', 'SORIA', 'SANTO DOMINGO DE LA CALZADA',
    # Cantabria
    'SANTANDER', 'TORRELAVEGA', 'LAREDO', 'PUENTE SAN MIGUEL',
    # País Vasco
    'BASAURI', 'SANTURCE', 'ZARAUZ', 'ARRASATE', 'LLODIO', 'ERANDIO',
    'ZUMAIA', 'LEGAZPI', 'SAN SEBASTIÁN', 'ALSASUA',
    # Navarra y La Rioja
    'PAMPLONA', 'AZAGRA', 'ARNEDO', 'LOGROÑO', 'CALAHORRA',
    # Aragón
    'HUESCA', 'ZARAGOZA', 'TARAZONA',
    # Cataluña
    'BARCELONA', 'RIPOLLET', 'EL PRAT DE LLOBREGAT', 'MOLINS DE REI',
    'ABRERA', 'OLOT', 'CORNELLA DE LLOBREGAT', 'IGUALADA', 'MANRESA',
    'LLORET DE MAR', 'L´AMETLLA DE MAR', "L'ESCALA", 'L’ ATMELLA DE MAR',
    'OLESA DE MONTSERRAT', 'CERVERA', 'SANT FOST DE CAMPSENTELLES',
    'TARRAGONA', 'LLEIDA', 'MANLLEU',
    # Comunidad Valenciana
    'XATIVA', 'ELCHE', 'BURRIANA', 'BENICARLO', 'GANDIA', 'ALTEA',
    'CULLERA', "VALL D'UXO", 'PATERNA', 'IBI', 'ALMANSA', 'MASSANASA',
    'CASTELLON', 'TEULADA', 'BENIFAYO', 'ONDA', 'MUSEROS', 'LA ELIANA',
    'ORIHUELA COSTA',
    # Baleares
    'PALMA DE MALLORCA', 'CAPDEPERA', 'MAÓ', 'PALMA', 'FERRERIES',
    'STA. EULÀRIA DES RIU',
    # Andalucía
    'ALMERIA', 'ANDUJAR', 'JAEN', 'MALAGA', 'MOTRIL', 'MOJACAR',
    'SEVILLA', 'EL EJIDO', 'ALHAURIN DE LA TORRE', 'SAN JOSE DE LA RINCONADA',
    # Murcia
    'LIBRILLA', 'CARTAGENA',
    # Castilla-La Mancha
    'CUENCA', 'VILLAREJO DE SALVANES', 'FUENSALIDA', 'LOS YEBENES',
    'TALAVERA DE LA REINA', 'SANTA CRUZ DE LA ZARZA', 'VALDEPEÑAS',
    # Madrid
    'MADRID', 'SAN LORENZO DEL ESCORIAL', 'COLLADO VILLALBA',
    'SAN SEBASTIAN DE LOS REYES', 'ALCALA DE HENARES', 'GETAFE',
    'ALCOBENDAS', 'SOTILLO DE LA ADRADA', 'ALALPARDO',
    # Extremadura
    'MONTIJO',
    # Otros
    'NOJA', 'AVI8LA',
}

# PORTUGAL — ciudades reconocibles
localidades_pt = {
    'LISBOA', 'MAFRA', 'PONTE DE LIMA', 'QUINTA DO CONDE',
    'CALDAS DAS TAIPAS', 'MACEDO DE CAVALEIROS',
    'VILA NOVA DE FAMALICAO', 'MONÇAO',
}

# BÉLGICA — ciudades reconocibles
localidades_be = {
    'EVERGEM', 'TEMSE', 'HERSELT', 'PUURS-KALFORT', 'BRUSSEL',
    'BILZEN', 'HULSTE', 'WINKSELE', 'BINCHE', 'TIENEN',
}

# PAÍSES BAJOS — ciudades reconocibles
localidades_nl = {
    'IJSSELSTEIN', 'PURMEREND', 'HHELEVOETSLUIS',
}

# ITALIA — ciudades reconocibles
localidades_it = {'ROMA'}

# ALEMANIA — ciudades reconocibles
localidades_de = {'FÜRTH'}

# ============================================================
# APLICAR CLASIFICACIÓN A LAS 178 LOCALIDADES
# ============================================================

clasificacion = []
for _, fila in sin_mapear.iterrows():
    loc = fila['localidad_upper']
    n = fila['num_clientes']
    if loc in localidades_es:
        pais = 'España'
    elif loc in localidades_pt:
        pais = 'Portugal'
    elif loc in localidades_be:
        pais = 'Bélgica'
    elif loc in localidades_nl:
        pais = 'Países Bajos'
    elif loc in localidades_it:
        pais = 'Italia'
    elif loc in localidades_de:
        pais = 'Alemania'
    else:
        pais = 'POR REVISAR'
    clasificacion.append({'localidad': loc, 'num_clientes': n, 'pais_detectado': pais})

import pandas as pd
df_clasif = pd.DataFrame(clasificacion)

# ============================================================
# RESUMEN
# ============================================================
print("=" * 70)
print("RESUMEN DE LA CLASIFICACIÓN AUTOMÁTICA")
print("=" * 70)
resumen = df_clasif.groupby('pais_detectado', as_index=False)['num_clientes'].sum()
resumen = resumen.sort_values('num_clientes', ascending=False)
print(resumen.to_string(index=False))
print(f"\nTotal: {df_clasif['num_clientes'].sum()} clientes")

# Mostrar los POR REVISAR para que los validemos
por_revisar = df_clasif[df_clasif['pais_detectado'] == 'POR REVISAR']
if len(por_revisar) > 0:
    print("\n" + "=" * 70)
    print("LOCALIDADES POR REVISAR (no reconocidas automáticamente)")
    print("=" * 70)
    print(por_revisar[['localidad', 'num_clientes']].to_string(index=False))

RESUMEN DE LA CLASIFICACIÓN AUTOMÁTICA
pais_detectado  num_clientes
       Bélgica             9
   POR REVISAR             9
        España             7
      Portugal             7
  Países Bajos             3
      Alemania             1
        Italia             1

Total: 37 clientes

LOCALIDADES POR REVISAR (no reconocidas automáticamente)
             localidad  num_clientes
             BOOISCHOT             1
VILA MOURA - QUARTEIRA             1
           DENDERLEEUW             1
              MARBELLA             1
              SABADELL             1
       RIBES DE FRESER             1
                  NULL             1
                   NaN             1
                 PORTO             1


In [10]:
print("=" * 70)
print("INVESTIGACIÓN DEL CLIENTE CON LOCALIDAD 'ND'")
print("=" * 70)

detalle_nd = con.execute("""
    SELECT 
        id_cliente,
        nombre_cliente,
        direccion_cliente,
        localidad_cliente,
        codigo_postal_original,
        codigo_provincia_cliente
    FROM silver.dim_cliente
    WHERE UPPER(TRIM(localidad_cliente)) = 'ND'
""").fetchdf()

print(detalle_nd.to_string(index=False))

INVESTIGACIÓN DEL CLIENTE CON LOCALIDAD 'ND'
id_cliente       nombre_cliente direccion_cliente localidad_cliente codigo_postal_original codigo_provincia_cliente
     31729 ABREU MARTINEZ, JOSE              VIGO                ND                  36315                     None
     32220         SIMON, ELENA    PTL VALLADARES                ND                  36315                     None


In [11]:
print("=" * 70)
print("DIAGNÓSTICO: clientes INTERNACIONAL con CP válido español")
print("=" * 70)

diagnostico = con.execute("""
    WITH base AS (
        SELECT 
            d.id_cliente,
            d.codigo_postal_original AS cp,
            d.codigo_provincia_cliente AS prov,
            d.localidad_cliente AS loc,
            -- ¿Tiene formato de CP español? (5 dígitos, primeros 2 entre 01 y 52)
            CASE
                WHEN LENGTH(TRIM(d.codigo_postal_original)) = 5
                 AND REGEXP_MATCHES(TRIM(d.codigo_postal_original), '^[0-9]+$')
                 AND CAST(SUBSTR(TRIM(d.codigo_postal_original), 1, 2) AS INTEGER) BETWEEN 1 AND 52
                THEN TRUE
                ELSE FALSE
            END AS cp_parece_espanol
        FROM silver.dim_cliente d
        LEFT JOIN silver.mapeo_paises m ON d.codigo_provincia_cliente = m.codigo
        WHERE d.tipo_mercado = 'INTERNACIONAL' AND m.pais IS NULL
    )
    SELECT 
        cp_parece_espanol,
        COUNT(*) AS num_clientes,
        SUM(CASE WHEN prov IS NULL THEN 1 ELSE 0 END) AS con_prov_null
    FROM base
    GROUP BY cp_parece_espanol
""").fetchdf()

print(diagnostico.to_string(index=False))

DIAGNÓSTICO: clientes INTERNACIONAL con CP válido español
 cp_parece_espanol  num_clientes  con_prov_null
              True             9            0.0
             False            28           27.0


In [12]:
print("=" * 70)
print("¿Los 'null' son NULL reales o el texto 'null'?")
print("=" * 70)

prueba = con.execute("""
    SELECT 
        codigo_provincia_cliente,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
      AND (codigo_provincia_cliente IS NULL 
           OR LOWER(TRIM(codigo_provincia_cliente)) IN ('null', 'nan', ''))
    GROUP BY codigo_provincia_cliente
    ORDER BY num_clientes DESC
""").fetchdf()

print(prueba.to_string(index=False))

¿Los 'null' son NULL reales o el texto 'null'?
codigo_provincia_cliente  num_clientes
                    None            27


In [13]:
import duckdb
from pathlib import Path

RUTA_DUCKDB = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark" / "duckdb" / "selmark.duckdb"
con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}\n")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb



In [14]:
print("=" * 70)
print("VALIDACIÓN 1 — VOLÚMENES Y CONSISTENCIA")
print("=" * 70)

resultado = con.execute("""
    SELECT 
        COUNT(*) AS total_clientes,
        SUM(CASE WHEN tipo_mercado = 'NACIONAL'      THEN 1 ELSE 0 END) AS nacionales,
        SUM(CASE WHEN tipo_mercado = 'INTERNACIONAL' THEN 1 ELSE 0 END) AS internacionales,
        SUM(CASE WHEN es_cliente_espanol = TRUE  THEN 1 ELSE 0 END) AS flag_espanol_true,
        SUM(CASE WHEN es_cliente_espanol = FALSE THEN 1 ELSE 0 END) AS flag_espanol_false
    FROM silver.dim_cliente
""").fetchdf()
print(resultado.T.to_string(header=False))

print("\n  ✓ Esperado: total = 3.492, nacionales ≈ 2.093, internacionales ≈ 1.399")
print("  ✓ es_cliente_espanol y tipo_mercado deben coincidir (NACIONAL ↔ TRUE)")

VALIDACIÓN 1 — VOLÚMENES Y CONSISTENCIA
total_clientes      3492.0
nacionales          2093.0
internacionales     1399.0
flag_espanol_true   2093.0
flag_espanol_false  1399.0

  ✓ Esperado: total = 3.469, nacionales ≈ 2.088, internacionales ≈ 1.381
  ✓ es_cliente_espanol y tipo_mercado deben coincidir (NACIONAL ↔ TRUE)


In [15]:
print("=" * 70)
print("VALIDACIÓN 2 — ¿Hay incoherencias entre es_cliente_espanol y tipo_mercado?")
print("=" * 70)

incoherencias = con.execute("""
    SELECT 
        es_cliente_espanol,
        tipo_mercado,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    GROUP BY es_cliente_espanol, tipo_mercado
    ORDER BY es_cliente_espanol DESC, tipo_mercado
""").fetchdf()
print(incoherencias.to_string(index=False))

print("\n  ✓ Solo deberían existir 2 combinaciones:")
print("    · TRUE  + NACIONAL")
print("    · FALSE + INTERNACIONAL")
print("  ❌ Si aparece TRUE+INTERNACIONAL o FALSE+NACIONAL, hay un bug en la lógica.")

VALIDACIÓN 2 — ¿Hay incoherencias entre es_cliente_espanol y tipo_mercado?
 es_cliente_espanol  tipo_mercado  num_clientes
               True      NACIONAL          2093
              False INTERNACIONAL          1399

  ✓ Solo deberían existir 2 combinaciones:
    · TRUE  + NACIONAL
    · FALSE + INTERNACIONAL
  ❌ Si aparece TRUE+INTERNACIONAL o FALSE+NACIONAL, hay un bug en la lógica.


In [16]:
print("=" * 70)
print("VALIDACIÓN 2 — ¿Hay incoherencias entre es_cliente_espanol y tipo_mercado?")
print("=" * 70)

incoherencias = con.execute("""
    SELECT 
        es_cliente_espanol,
        tipo_mercado,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    GROUP BY es_cliente_espanol, tipo_mercado
    ORDER BY es_cliente_espanol DESC, tipo_mercado
""").fetchdf()
print(incoherencias.to_string(index=False))

print("\n  ✓ Solo deberían existir 2 combinaciones:")
print("    · TRUE  + NACIONAL")
print("    · FALSE + INTERNACIONAL")
print("  ❌ Si aparece TRUE+INTERNACIONAL o FALSE+NACIONAL, hay un bug en la lógica.")

VALIDACIÓN 2 — ¿Hay incoherencias entre es_cliente_espanol y tipo_mercado?
 es_cliente_espanol  tipo_mercado  num_clientes
               True      NACIONAL          2093
              False INTERNACIONAL          1399

  ✓ Solo deberían existir 2 combinaciones:
    · TRUE  + NACIONAL
    · FALSE + INTERNACIONAL
  ❌ Si aparece TRUE+INTERNACIONAL o FALSE+NACIONAL, hay un bug en la lógica.


In [17]:
print("=" * 70)
print("VALIDACIÓN 3 — CALIDAD DEL CP EN CLIENTES NACIONALES")
print("=" * 70)

cp_nacional = con.execute("""
    SELECT 
        COUNT(*) AS total_nacionales,
        SUM(CASE WHEN codigo_postal_norm IS NULL THEN 1 ELSE 0 END) AS cp_nulo,
        SUM(CASE WHEN LENGTH(codigo_postal_norm) = 5 
                 AND REGEXP_MATCHES(codigo_postal_norm, '^[0-9]+$') THEN 1 ELSE 0 END) AS cp_valido_5digitos,
        SUM(CASE WHEN LENGTH(codigo_postal_norm) <> 5 
                 OR NOT REGEXP_MATCHES(codigo_postal_norm, '^[0-9]+$') THEN 1 ELSE 0 END) AS cp_anomalo,
        COUNT(DISTINCT codigo_postal_norm) AS cps_distintos
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'NACIONAL'
""").fetchdf()
print(cp_nacional.T.to_string(header=False))

print("\n  ✓ TODOS los clientes nacionales deben tener CP válido de 5 dígitos.")
print("  ❌ Si cp_nulo o cp_anomalo > 0, hay clientes nacionales que no podrán cruzarse con MOSAIC.")

VALIDACIÓN 3 — CALIDAD DEL CP EN CLIENTES NACIONALES
total_nacionales    2093.0
cp_nulo                0.0
cp_valido_5digitos  2093.0
cp_anomalo             0.0
cps_distintos       1210.0

  ✓ TODOS los clientes nacionales deben tener CP válido de 5 dígitos.
  ❌ Si cp_nulo o cp_anomalo > 0, hay clientes nacionales que no podrán cruzarse con MOSAIC.


In [18]:
print("=" * 70)
print("VALIDACIÓN 4 — COBERTURA CON silver.mosaic (la clave del geomarketing)")
print("=" * 70)

cobertura = con.execute("""
    SELECT 
        COUNT(*) AS clientes_nacionales,
        SUM(CASE WHEN m.codigo_postal_norm IS NOT NULL THEN 1 ELSE 0 END) AS con_mosaic,
        SUM(CASE WHEN m.codigo_postal_norm IS NULL THEN 1 ELSE 0 END) AS sin_mosaic,
        ROUND(
            100.0 * SUM(CASE WHEN m.codigo_postal_norm IS NOT NULL THEN 1 ELSE 0 END) 
            / COUNT(*), 2
        ) AS pct_cobertura
    FROM silver.dim_cliente d
    LEFT JOIN silver.mosaic m ON d.codigo_postal_norm = m.codigo_postal_norm
    WHERE d.tipo_mercado = 'NACIONAL'
""").fetchdf()
print(cobertura.T.to_string(header=False))

print("\n  ✓ Esperado: cobertura ≥ 95 %")
print("  ⚠ Si baja del 90 %, conviene revisar los clientes sin MOSAIC")

VALIDACIÓN 4 — COBERTURA CON silver.mosaic (la clave del geomarketing)
clientes_nacionales  2093.00
con_mosaic           2035.00
sin_mosaic             58.00
pct_cobertura          97.23

  ✓ Esperado: cobertura ≥ 95 %
  ⚠ Si baja del 90 %, conviene revisar los clientes sin MOSAIC


In [19]:
print("=" * 70)
print("VALIDACIÓN 5 — CLIENTES NACIONALES QUE NO CRUZAN CON MOSAIC")
print("=" * 70)

sin_mosaic = con.execute("""
    SELECT 
        d.id_cliente,
        d.nombre_cliente,
        d.localidad_cliente,
        d.codigo_postal_norm,
        d.codigo_provincia_cliente
    FROM silver.dim_cliente d
    LEFT JOIN silver.mosaic m ON d.codigo_postal_norm = m.codigo_postal_norm
    WHERE d.tipo_mercado = 'NACIONAL' AND m.codigo_postal_norm IS NULL
    ORDER BY d.codigo_postal_norm
    LIMIT 30
""").fetchdf()

if len(sin_mosaic) == 0:
    print("  ✅ ¡Perfecto! Todos los nacionales cruzan con MOSAIC.")
else:
    print(f"  ⚠ {len(sin_mosaic)} clientes nacionales sin MOSAIC (mostrando primeros 30):")
    print(sin_mosaic.to_string(index=False))

VALIDACIÓN 5 — CLIENTES NACIONALES QUE NO CRUZAN CON MOSAIC
  ⚠ 30 clientes nacionales sin MOSAIC (mostrando primeros 30):
id_cliente                    nombre_cliente       localidad_cliente codigo_postal_norm codigo_provincia_cliente
      7498    MUNUERA SANCHEZ, MARIA EUGENIA              TORREVIEJA              03181                       03
     32071            CACHUTT MOLINA, NIURKA             AGUA AMARGA              04149                       04
      3815                   MANGOLINE, S.C.                   MAHON              07701                       07
      7342           GOMILA TRIAY, MARGARITA                     MAO              07701                       07
      6591                CA NA GENERA, S.L.               FERRERIES              07750                      NaN
     31098          PULIDO VISIEDO, FRANCINA                 MANRESA              08242                      NaN
      6798              FABREGA ROSAS, PILAR              GRANOLLERS              0840

In [20]:
print("=" * 70)
print("VALIDACIÓN 6 — DISTRIBUCIÓN POR PROVINCIA (TOP 10)")
print("=" * 70)

por_provincia = con.execute("""
    SELECT 
        SUBSTR(codigo_postal_norm, 1, 2) AS provincia_cp,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'NACIONAL'
    GROUP BY provincia_cp
    ORDER BY num_clientes DESC
    LIMIT 10
""").fetchdf()
print(por_provincia.to_string(index=False))

print("\n  ✓ Esperado: '36' (Pontevedra) en el top — es la sede de Selmark.")
print("  ✓ Otras provincias típicas en moda íntima: 28 (Madrid), 08 (Barcelona),")
print("    46 (Valencia), 41 (Sevilla), 29 (Málaga), 15 (A Coruña).")

VALIDACIÓN 6 — DISTRIBUCIÓN POR PROVINCIA (TOP 10)
provincia_cp  num_clientes
          08           242
          36           159
          33           105
          28           102
          15            96
          46            92
          03            60
          38            57
          43            57
          20            55

  ✓ Esperado: '36' (Pontevedra) en el top — es la sede de Selmark.
  ✓ Otras provincias típicas en moda íntima: 28 (Madrid), 08 (Barcelona),
    46 (Valencia), 41 (Sevilla), 29 (Málaga), 15 (A Coruña).


In [21]:
print("=" * 70)
print("VALIDACIÓN 7 — COBERTURA DEL MAPEO DE PAÍSES EN INTERNACIONALES")
print("=" * 70)

mapeo_internacional = con.execute("""
    SELECT 
        COALESCE(m.pais, 'SIN MAPEAR') AS pais,
        COUNT(*) AS num_clientes
    FROM silver.dim_cliente d
    LEFT JOIN silver.mapeo_paises m ON d.codigo_provincia_cliente = m.codigo
    WHERE d.tipo_mercado = 'INTERNACIONAL'
    GROUP BY pais
    ORDER BY num_clientes DESC
""").fetchdf()
print(mapeo_internacional.to_string(index=False))

# Resumen
total_int = mapeo_internacional['num_clientes'].sum()
sin_mapear = mapeo_internacional.loc[mapeo_internacional['pais'] == 'SIN MAPEAR', 'num_clientes']
sin_mapear_n = int(sin_mapear.iloc[0]) if len(sin_mapear) > 0 else 0
print(f"\n  Total internacionales: {total_int}")
print(f"  Sin mapear: {sin_mapear_n}  ({100*sin_mapear_n/total_int:.2f} %)")
print("\n  ✓ Esperado: ≤ 30 clientes sin mapear (los pocos que no tienen CP español ni")
print("    código de provincia reconocido).")

VALIDACIÓN 7 — COBERTURA DEL MAPEO DE PAÍSES EN INTERNACIONALES
                  pais  num_clientes
                Italia           632
              Portugal           337
               Polonia            84
               Bélgica            61
               Noruega            56
                Canadá            55
          Países Bajos            46
            SIN MAPEAR            37
  España (error carga)            10
                 Rusia             7
           Reino Unido             6
              Alemania             5
                México             4
               Francia             3
                 China             2
                 Malta             2
                 Chile             2
               Austria             2
                 India             2
               Andorra             2
             Sudáfrica             2
  Bélgica/Países Bajos             2
                Israel             2
        Estados Unidos             2
           

In [22]:
print("=" * 70)
print("RESUMEN — ¿ESTAMOS LISTOS PARA EL GEOMARKETING?")
print("=" * 70)

resumen = con.execute("""
    SELECT 
        'Total clientes en silver.dim_cliente' AS metrica,
        COUNT(*) AS valor
    FROM silver.dim_cliente
    UNION ALL
    SELECT 'Clientes nacionales', 
           COUNT(*) FROM silver.dim_cliente WHERE tipo_mercado = 'NACIONAL'
    UNION ALL
    SELECT 'Nacionales con CP válido y match en MOSAIC',
           COUNT(*) 
           FROM silver.dim_cliente d
           INNER JOIN silver.mosaic m ON d.codigo_postal_norm = m.codigo_postal_norm
           WHERE d.tipo_mercado = 'NACIONAL'
    UNION ALL
    SELECT 'Internacionales con país identificado',
           COUNT(*) 
           FROM silver.dim_cliente d
           INNER JOIN silver.mapeo_paises mp ON d.codigo_provincia_cliente = mp.codigo
           WHERE d.tipo_mercado = 'INTERNACIONAL'
""").fetchdf()
print(resumen.to_string(index=False))

RESUMEN — ¿ESTAMOS LISTOS PARA EL GEOMARKETING?
                                   metrica  valor
      Total clientes en silver.dim_cliente   3492
                       Clientes nacionales   2093
Nacionales con CP válido y match en MOSAIC   2035
     Internacionales con país identificado   1362


In [23]:
print("=" * 70)
print("DIAGNÓSTICO — 10 clientes con código español pero clasificados internacional")
print("=" * 70)

bichos_raros = con.execute("""
    SELECT 
        d.id_cliente,
        d.nombre_cliente,
        d.localidad_cliente,
        d.codigo_postal_original,
        d.codigo_postal_norm,
        d.codigo_provincia_cliente,
        d.tipo_mercado
    FROM silver.dim_cliente d
    INNER JOIN silver.mapeo_paises m ON d.codigo_provincia_cliente = m.codigo
    WHERE m.pais = 'España (error carga)'
    ORDER BY d.codigo_provincia_cliente, d.id_cliente
""").fetchdf()
print(bichos_raros.to_string(index=False))

DIAGNÓSTICO — 10 clientes con código español pero clasificados internacional
id_cliente                                  nombre_cliente               localidad_cliente codigo_postal_original codigo_postal_norm codigo_provincia_cliente  tipo_mercado
         1                         PEREZ RODRIGUEZ, AMADOR                            VIGO                  36203              36203                      036      NACIONAL
      1964                                 SELMARK, S.L.U.                            VIGO                  36315              36315                      036      NACIONAL
       242                       BASTOS COSTAS, MARIA JOSÉ                            VIGO                  36213              36213                      036      NACIONAL
       248                          CARRERA LAGO, VERONICA             LOUREDOSAN SALVADOR                  36415              36415                      036      NACIONAL
       256                     DACOSTA ALVAREZ, MONTSERRAT     

In [24]:
print("=" * 70)
print("DIAGNÓSTICO — 10 clientes con código español pero clasificados internacional")
print("=" * 70)

bichos_raros = con.execute("""
    SELECT 
        d.id_cliente,
        d.nombre_cliente,
        d.localidad_cliente,
        d.codigo_postal_original,
        d.codigo_postal_norm,
        d.codigo_provincia_cliente,
        d.tipo_mercado
    FROM silver.dim_cliente d
    INNER JOIN silver.mapeo_paises m ON d.codigo_provincia_cliente = m.codigo
    WHERE m.pais = 'España (error carga)'
    ORDER BY d.codigo_provincia_cliente, d.id_cliente
""").fetchdf()
print(bichos_raros.to_string(index=False))

DIAGNÓSTICO — 10 clientes con código español pero clasificados internacional
id_cliente                                  nombre_cliente               localidad_cliente codigo_postal_original codigo_postal_norm codigo_provincia_cliente  tipo_mercado
         1                         PEREZ RODRIGUEZ, AMADOR                            VIGO                  36203              36203                      036      NACIONAL
      1964                                 SELMARK, S.L.U.                            VIGO                  36315              36315                      036      NACIONAL
       242                       BASTOS COSTAS, MARIA JOSÉ                            VIGO                  36213              36213                      036      NACIONAL
       248                          CARRERA LAGO, VERONICA             LOUREDOSAN SALVADOR                  36415              36415                      036      NACIONAL
       256                     DACOSTA ALVAREZ, MONTSERRAT     

## 8. Conclusiones del notebook 03b

### Resumen de la investigación

Esta investigación complementaria al notebook 03 ha permitido caracterizar la composición geográfica de la cartera internacional de Selmark y construir la tabla auxiliar `silver.mapeo_paises`, que servirá de soporte para los análisis de cartera internacional en las capas superiores del modelo.

### Hallazgos principales

La cartera de clientes de Selmark, materializada en la tabla `silver.dim_cliente`, contiene 3.492 registros (2.093 nacionales y 1.399 internacionales) tras la aplicación de las tres reglas de clasificación definidas en el notebook 03. Sobre este punto de partida, el notebook 03b se ha centrado exclusivamente en la caracterización de la cartera internacional, sin introducir modificaciones adicionales sobre `silver.dim_cliente`.

A partir del análisis de las localidades y de los patrones del código postal asociados a los códigos de provincia distintos presentes en la cartera internacional, se ha construido la tabla `silver.mapeo_paises`, que asocia cada código de provincia a un país identificado. La cobertura efectiva del mapeo alcanza el 97,36 % de la cartera internacional, con 1.362 de los 1.399 clientes correctamente identificados con su país de origen.

La distribución por país revela una concentración significativa en cuatro mercados europeos (Italia con 632 clientes, Portugal con 337, Polonia con 84 y Bélgica con 61), que en conjunto suman el 79,7 % de la cartera internacional. El resto se distribuye en una larga cola de mercados con presencia menor.

Sobre la cartera nacional, el cruce con la tabla `silver.mosaic` por código postal alcanza una cobertura del 97,23 %, con 2.035 de los 2.093 clientes nacionales emparejados con información sociodemográfica de Experian. Esta cobertura resulta plenamente suficiente para garantizar la representatividad de los análisis de geomarketing posteriores.

### Casos residuales documentados

- 58 clientes nacionales sin enriquecimiento MOSAIC, correspondientes a códigos postales españoles no cubiertos por la base de datos de Experian.
- 37 clientes internacionales sin país identificado, asociados a códigos de provincia que la investigación no ha podido vincular a un país concreto por insuficiencia de información en los campos auxiliares (localidad, dirección y código postal).

Estos casos residuales se mantienen en la tabla con su clasificación actual para preservar la trazabilidad completa del maestro. Su volumen agregado (95 clientes, equivalentes al 2,72 % del total) resulta despreciable y no afecta al rigor metodológico de los análisis posteriores.

### Tablas resultantes

La ejecución del notebook deja disponibles en la capa Silver las siguientes estructuras:

- `silver.dim_cliente`: maestro de clientes con 3.492 registros (2.093 nacionales y 1.399 internacionales), sin modificaciones respecto a la versión generada por el notebook 03.
- `silver.mapeo_paises`: tabla auxiliar con el mapeo entre código de provincia y país, que cubre 53 países distintos.

In [25]:
con.close()
print("✅ Conexión cerrada. Investigación complementaria al notebook 03 finalizada.")
print("\n📋 Tablas actualizadas en silver:")
print("   · silver.dim_cliente   (refinada con 3 reglas: A + B + C)")
print("   · silver.mapeo_paises  (nueva tabla auxiliar)")

✅ Conexión cerrada. Investigación complementaria al notebook 03 finalizada.

📋 Tablas actualizadas en silver:
   · silver.dim_cliente   (refinada con 3 reglas: A + B + C)
   · silver.mapeo_paises  (nueva tabla auxiliar)
